# Questão 6 — Previsão de Demanda

## Objetivo

Construir um baseline para prever a demanda mensal do produto
`Bússola de Bordo 702` no primeiro trimestre de 2026.

O modelo utiliza a média móvel das vendas dos três meses imediatamente
anteriores à data prevista.

### Divisão temporal

- Treino: dados até 31/12/2025;
- Teste: janeiro, fevereiro e março de 2026;
- Granularidade: mensal;
- Métrica de avaliação: MAE (Mean Absolute Error).

### Estratégia

1. Relacionar produtos, variantes, pedidos e itens de pedidos;
2. Agregar a quantidade vendida por mês;
3. Criar uma série mensal contínua, preenchendo meses sem vendas com zero;
4. Calcular a média móvel dos três meses anteriores;
5. Comparar as previsões com as vendas reais do primeiro trimestre de 2026.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd


PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

conn = duckdb.connect()

In [2]:
tables = [
    "products",
    "product_variants",
    "orders",
    "order_items",
]

for table in tables:
    csv_path = RAW_DATA_DIR / f"{table}.csv"

    conn.execute(
        f"""
        CREATE OR REPLACE TABLE {table} AS
        SELECT *
        FROM read_csv_auto('{csv_path.as_posix()}');
        """
    )

print("Tabelas carregadas com sucesso.")

Tabelas carregadas com sucesso.


In [3]:
query_sales = """
SELECT
    CAST(DATE_TRUNC('month', o.placed_at) AS DATE) AS mes,
    p.id AS product_id,
    p.name AS produto,
    pv.id AS product_variant_id,
    oi.order_id,
    oi.quantity
FROM products AS p
INNER JOIN product_variants AS pv
    ON pv.product_id = p.id
INNER JOIN order_items AS oi
    ON oi.product_variant_id = pv.id
INNER JOIN orders AS o
    ON o.id = oi.order_id
WHERE p.name = 'Bússola de Bordo 702'
ORDER BY mes;
"""

product_sales = conn.execute(query_sales).df()

product_sales.head()

,mes,product_id,produto,product_variant_id,order_id,quantity
0,2020-01-01,240,Bússola de Bordo 702,486,666,9
1,2020-01-01,74,Bússola de Bordo 702,147,3445,6
2,2020-01-01,74,Bússola de Bordo 702,147,11381,4
3,2020-01-01,74,Bússola de Bordo 702,148,21540,9
4,2020-01-01,74,Bússola de Bordo 702,147,46233,1


In [4]:
query_monthly_demand = """
WITH product_sales AS (
    SELECT
        CAST(DATE_TRUNC('month', o.placed_at) AS DATE) AS mes,
        SUM(oi.quantity) AS unidades_vendidas
    FROM products AS p
    INNER JOIN product_variants AS pv
        ON pv.product_id = p.id
    INNER JOIN order_items AS oi
        ON oi.product_variant_id = pv.id
    INNER JOIN orders AS o
        ON o.id = oi.order_id
    WHERE p.name = 'Bússola de Bordo 702'
    GROUP BY DATE_TRUNC('month', o.placed_at)
),

calendar AS (
    SELECT
        CAST(month_date AS DATE) AS mes
    FROM GENERATE_SERIES(
        (SELECT MIN(mes) FROM product_sales),
        DATE '2026-03-01',
        INTERVAL 1 MONTH
    ) AS dates(month_date)
)

SELECT
    c.mes,
    COALESCE(ps.unidades_vendidas, 0) AS unidades_vendidas
FROM calendar AS c
LEFT JOIN product_sales AS ps
    ON ps.mes = c.mes
ORDER BY c.mes;
"""

monthly_demand = conn.execute(query_monthly_demand).df()

monthly_demand.tail(15)

,mes,unidades_vendidas
60,2025-01-01,57.0
61,2025-02-01,32.0
62,2025-03-01,77.0
63,2025-04-01,38.0
64,2025-05-01,24.0
65,2025-06-01,17.0
66,2025-07-01,19.0
67,2025-08-01,23.0
68,2025-09-01,31.0
69,2025-10-01,34.0


In [5]:
train = monthly_demand[
    monthly_demand["mes"] <= pd.Timestamp("2025-12-31")
].copy()

test = monthly_demand[
    (monthly_demand["mes"] >= pd.Timestamp("2026-01-01"))
    & (monthly_demand["mes"] <= pd.Timestamp("2026-03-31"))
].copy()

print(f"Meses de treino: {len(train)}")
print(f"Meses de teste: {len(test)}")

Meses de treino: 72
Meses de teste: 3


In [7]:
monthly_demand["previsao"] = (
    monthly_demand["unidades_vendidas"]
    .shift(1)
    .rolling(window=3)
    .mean()
)

In [8]:
forecast = monthly_demand.loc[
    (monthly_demand["mes"] >= "2026-01-01")
    & (monthly_demand["mes"] <= "2026-03-31"),
    ["mes", "unidades_vendidas", "previsao"],
].copy()

forecast

,mes,unidades_vendidas,previsao
72,2026-01-01,79.0,38.666667
73,2026-02-01,68.0,53.666667
74,2026-03-01,60.0,56.333333


In [9]:
forecast["erro_absoluto"] = (
    forecast["unidades_vendidas"] - forecast["previsao"]
).abs()

mae = forecast["erro_absoluto"].mean()

print(f"MAE: {mae:.2f} unidades")

MAE: 19.44 unidades


In [10]:
forecast

,mes,unidades_vendidas,previsao,erro_absoluto
72,2026-01-01,79.0,38.666667,40.333333
73,2026-02-01,68.0,53.666667,14.333333
74,2026-03-01,60.0,56.333333,3.666667


A avaliação foi realizada de forma walk-forward: para cada mês do período
de teste, a previsão utiliza exclusivamente os três meses anteriores.
Assim, a previsão de fevereiro pode utilizar a demanda real observada em
janeiro, e a previsão de março pode utilizar os valores reais de janeiro
e fevereiro, pois esses dados já são conhecidos nas respectivas datas
de previsão.